# Segmentation et analyse de zones de soudure (U-Net + CNN)

## Cahier de charges (résumé)
- **Objectif**: identifier automatiquement les zones de soudure (images/vidéos) et évaluer **largeur**, **continuité**, **homogénéité** pour faciliter le contrôle qualité.
- **IA utilisée**:
  - **U-Net** (segmentation) → masque de soudure
  - **CNN simple** (classification) → homogène / non homogène
- **Entrées**: images/vidéos, éclairage standardisé, métadonnées caméra (angle, résolution).
- **Sorties**: masques segmentés, largeur & continuité estimées, score qualité/homogénéité.
- **Dataset recommandé**: images industrielles publiques de soudures (ex: **GDXray**).

> Note importante: de nombreux jeux de données « industriels » ne fournissent pas de masques de segmentation. Ce notebook gère **2 cas**:
> 1) Dataset avec **masques** (supervisé)
> 2) Dataset sans masques: génération de **pseudo-masques** par traitement d’image (weak/self-supervised) puis entraînement U-Net dessus.


In [ ]:
# --- Setup (Colab) ---
# Exécuter cette cellule en premier.

import os
import sys
import platform

print('Python:', sys.version)
print('Platform:', platform.platform())

# Install deps (silencieux)
!pip -q install --upgrade pip
!pip -q install albumentations==1.4.21 opencv-python==4.10.0.84 matplotlib==3.8.4 tqdm==4.66.5
!pip -q install segmentation-models-pytorch==0.3.3 timm==1.0.9
!pip -q install scikit-image==0.24.0 ipywidgets==8.1.5
!pip -q install kaggle==1.6.17 pycocotools==2.0.7

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Colab widgets
from google.colab import output
output.enable_custom_widget_manager()


## 1) Télécharger le dataset (d’abord) via Kaggle + organisation

Ce notebook télécharge **automatiquement** le dataset complet via **Kaggle**.

### Dataset choisi (conforme au cahier de charges)
- **Weld Quality Inspection – Instance Segmentation** (annotations de segmentation)
- **Kaggle slug**: `sukmaadhiwijaya/weld-quality-inspection-instance-segmentation`
- **Page Kaggle**: `https://www.kaggle.com/datasets/sukmaadhiwijaya/weld-quality-inspection-instance-segmentation`

Ce notebook est conçu pour fonctionner avec:
- **Dataset annoté** (COCO/instance seg ou masques) → entraînement U‑Net supervisé
- **Dataset non annoté** (fallback) → génération de **pseudo-masques** (section suivante)

Pré-requis:
- Avoir des identifiants Kaggle (`kaggle.json` ou variables d'environnement).


In [ ]:
# --- Dataset download & layout (Kaggle: auto) ---
# Cette section télécharge automatiquement le dataset complet via Kaggle.
# Pré-requis Kaggle:
# - Option A (recommandée): uploader `kaggle.json` dans /content/kaggle.json
# - Option B: définir variables d'environnement `KAGGLE_USERNAME` et `KAGGLE_KEY`

from pathlib import Path
import shutil
import tarfile
import zipfile
import os
import json
from collections import defaultdict
from typing import Optional

import numpy as np
import cv2

try:
    from pycocotools import mask as mask_utils
except Exception as e:
    mask_utils = None
    print('pycocotools not available yet:', e)

DATA_ROOT = Path('/content/weld_dataset')
DOWNLOADS_DIR = DATA_ROOT / 'downloads'
EXTRACT_DIR = DATA_ROOT / 'extracted'
IMAGES_DIR = DATA_ROOT / 'images'
MASKS_DIR = DATA_ROOT / 'masks'  # optionnel

DATA_ROOT.mkdir(parents=True, exist_ok=True)
DOWNLOADS_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MASKS_DIR.mkdir(parents=True, exist_ok=True)

# >>> Dataset Kaggle (auto)
# Ce notebook utilise un dataset de soudure AVEC annotations de segmentation.
# (slug Kaggle)
KAGGLE_DATASET = 'sukmaadhiwijaya/weld-quality-inspection-instance-segmentation'

# Si votre dataset est un "competition dataset" au lieu d'un "dataset",
# remplacez la commande plus bas par `kaggle competitions download -c <competition>`.


def ensure_kaggle_auth() -> None:
    """Configure l'auth Kaggle (sans interaction si possible)."""
    kaggle_dir = Path('/root/.kaggle')
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / 'kaggle.json'

    # 1) Use uploaded kaggle.json
    uploaded = Path('/content/kaggle.json')
    if uploaded.exists():
        shutil.copy2(uploaded, target)
        os.chmod(target, 0o600)
        return

    # 2) Use env vars
    user = os.environ.get('KAGGLE_USERNAME')
    key = os.environ.get('KAGGLE_KEY')
    if user and key:
        target.write_text('{"username": "%s", "key": "%s"}' % (user, key))
        os.chmod(target, 0o600)
        return

    # 3) As a last resort, ask user to upload (Colab)
    try:
        from google.colab import files
        print('Kaggle credentials not found. Please upload kaggle.json now...')
        files.upload()  # user uploads kaggle.json
        if uploaded.exists():
            shutil.copy2(uploaded, target)
            os.chmod(target, 0o600)
            return
    except Exception:
        pass

    raise FileNotFoundError(
        'Kaggle credentials not found. Upload kaggle.json to /content/kaggle.json '
        'or set KAGGLE_USERNAME and KAGGLE_KEY env vars.'
    )


def find_coco_json(root: Path) -> Optional[Path]:
    """Trouve un fichier COCO (images+annotations) dans l'archive extraite."""
    for p in sorted(root.rglob('*.json')):
        try:
            obj = json.loads(p.read_text())
        except Exception:
            continue
        if isinstance(obj, dict) and 'images' in obj and 'annotations' in obj:
            return p
    return None


def coco_union_mask(coco: dict, image_id: int, height: int, width: int) -> np.ndarray:
    """Construit un masque binaire union (H,W) à partir des annotations COCO."""
    m = np.zeros((height, width), dtype=np.uint8)
    anns = coco['_ann_by_image'].get(image_id, [])
    if not anns:
        return m

    if mask_utils is None:
        raise RuntimeError('pycocotools is required to build masks from COCO. Install pycocotools.')

    for ann in anns:
        seg = ann.get('segmentation')
        if seg is None:
            continue
        # Polygons
        if isinstance(seg, list):
            rles = mask_utils.frPyObjects(seg, height, width)
            rle = mask_utils.merge(rles)
            dec = mask_utils.decode(rle)
        # RLE
        elif isinstance(seg, dict) and 'counts' in seg:
            dec = mask_utils.decode(seg)
        else:
            continue

        if dec is None:
            continue
        if dec.ndim == 3:
            dec = np.any(dec, axis=2)
        m = np.maximum(m, (dec.astype(np.uint8) * 255))

    return m


def extract_archive(archive_path: Path) -> None:
    print('Extracting:', archive_path.name)

    if archive_path.suffix.lower() == '.zip' or zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        return

    if tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path, 'r:*') as tf:
            tf.extractall(EXTRACT_DIR)
        return

    raise ValueError(f"Unknown archive format: {archive_path}")


def _looks_like_image(p: Path) -> bool:
    return p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}


def organize_dataset() -> None:
    """Organise images + masks dans DATA_ROOT.

    Priorité:
    1) **COCO instance segmentation** (annotations json) → conversion en masques binaires
    2) sinon: heuristique dossiers images/ + masks/
    """
    # Clear target dirs
    for d in (IMAGES_DIR, MASKS_DIR):
        for f in d.glob('*'):
            if f.is_file():
                f.unlink()

    # 1) COCO (recommended)
    coco_path = find_coco_json(EXTRACT_DIR)
    if coco_path is not None:
        print('COCO annotations found:', coco_path)
        coco = json.loads(coco_path.read_text())
        coco['_ann_by_image'] = defaultdict(list)
        for ann in coco.get('annotations', []):
            if 'image_id' in ann:
                coco['_ann_by_image'][ann['image_id']].append(ann)

        images = coco.get('images', [])
        if not images:
            raise RuntimeError('COCO json has no images[]')

        # Sort by id for stable ordering
        images = sorted(images, key=lambda x: x.get('id', 0))

        for i, im in enumerate(images):
            file_name = im.get('file_name')
            h = int(im.get('height'))
            w = int(im.get('width'))
            image_id = int(im.get('id'))
            if not file_name:
                continue

            src = EXTRACT_DIR / file_name
            if not src.exists():
                # fallback: search by basename
                hits = list(EXTRACT_DIR.rglob(Path(file_name).name))
                if not hits:
                    continue
                src = hits[0]

            out_img = IMAGES_DIR / f"img_{i:06d}{src.suffix.lower()}"
            shutil.copy2(src, out_img)

            m = coco_union_mask(coco, image_id=image_id, height=h, width=w)
            out_msk = MASKS_DIR / f"mask_{i:06d}.png"
            cv2.imwrite(out_msk.as_posix(), m)

        print('Images:', len(list(IMAGES_DIR.glob('*'))))
        print('Masks :', len(list(MASKS_DIR.glob('*'))))
        return

    # 2) Heuristique dossiers
    candidates = list(EXTRACT_DIR.rglob('*'))
    img_roots = [p for p in candidates if p.is_dir() and p.name.lower() in {'images', 'imgs', 'image'}]
    msk_roots = [p for p in candidates if p.is_dir() and p.name.lower() in {'masks', 'mask', 'labels', 'annotations'}]

    if img_roots:
        src = img_roots[0]
        imgs = [p for p in src.rglob('*') if p.is_file() and _looks_like_image(p)]
    else:
        imgs = [p for p in EXTRACT_DIR.rglob('*') if p.is_file() and _looks_like_image(p)]

    if not imgs:
        raise RuntimeError('No images found after Kaggle extraction. Check dataset contents.')

    for i, p in enumerate(sorted(imgs)):
        out = IMAGES_DIR / f"img_{i:06d}{p.suffix.lower()}"
        shutil.copy2(p, out)

    if msk_roots:
        src = msk_roots[0]
        msks = [p for p in src.rglob('*') if p.is_file() and _looks_like_image(p)]
        for i, p in enumerate(sorted(msks)):
            out = MASKS_DIR / f"mask_{i:06d}{p.suffix.lower()}"
            shutil.copy2(p, out)

    print('Images:', len(list(IMAGES_DIR.glob('*'))))
    print('Masks :', len(list(MASKS_DIR.glob('*'))))


# --- Run Kaggle download (full dataset first) ---
if not KAGGLE_DATASET:
    raise ValueError('Set KAGGLE_DATASET (e.g. "username/dataset-name").')

ensure_kaggle_auth()

# Download only if not already extracted for THIS dataset
sentinel = EXTRACT_DIR / '.extracted_ok'
need_extract = True
if sentinel.exists():
    try:
        need_extract = (sentinel.read_text().strip() != KAGGLE_DATASET)
    except Exception:
        need_extract = True

if need_extract:
    # Clean extract dir
    if EXTRACT_DIR.exists():
        for p in EXTRACT_DIR.glob('*'):
            if p.is_file():
                p.unlink()
            else:
                shutil.rmtree(p)

    # Kaggle API downloads a zip named after dataset slug
    dataset_name = KAGGLE_DATASET.split('/')[-1]
    out_zip = DOWNLOADS_DIR / f"{dataset_name}.zip"

    if not out_zip.exists() or out_zip.stat().st_size == 0:
        !kaggle datasets download -d "{KAGGLE_DATASET}" -p "{DOWNLOADS_DIR}"

    if not out_zip.exists():
        # fallback: take the newest zip in downloads
        zips = sorted(DOWNLOADS_DIR.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
        if not zips:
            raise FileNotFoundError('Kaggle download did not produce a .zip in downloads/')
        out_zip = zips[0]

    extract_archive(out_zip)
    sentinel.write_text(KAGGLE_DATASET)

organize_dataset()


### Kaggle (auto)

Le téléchargement est maintenant **automatique** dans la cellule précédente.

Pré-requis:
- **Uploader** `kaggle.json` dans `/content/kaggle.json` **ou** définir `KAGGLE_USERNAME` et `KAGGLE_KEY`.
- Renseigner `KAGGLE_DATASET` (slug Kaggle: `username/dataset-name`).


In [ ]:
# (Deprecated) Cette cellule n'est plus nécessaire.
# Le téléchargement Kaggle est géré automatiquement dans la cellule "Dataset download & layout".
print('Kaggle download is handled automatically above. You can skip this cell.')


## 2) Masques: utiliser ceux du dataset OU générer des pseudo-masques

Si `masks/` est vide, on génère des **pseudo-masques** à partir des images (contraste + top-hat + seuillage + morphologie). Ces masques servent de labels faibles pour entraîner U‑Net.


In [ ]:
import cv2
import numpy as np
from tqdm import tqdm


def generate_pseudo_mask(gray: np.ndarray) -> np.ndarray:
    """Retourne un masque binaire (uint8 0/255) de la soudure."""
    # Normalisation + CLAHE
    gray = gray.astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g = clahe.apply(gray)

    # Top-hat pour faire ressortir une bande brillante/contrastée
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    tophat = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, kernel)

    # Lissage + Otsu
    blur = cv2.GaussianBlur(tophat, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Nettoyage morpho
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k2, iterations=2)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, k2, iterations=1)

    # Garder la(les) composante(s) principale(s)
    num, lab, stats, _ = cv2.connectedComponentsWithStats(th, connectivity=8)
    if num <= 1:
        return th

    # Trier par aire (ignorer fond=0)
    areas = stats[1:, cv2.CC_STAT_AREA]
    order = np.argsort(-areas)
    keep = order[:1]  # garder 1 composante (modifiable)

    out = np.zeros_like(th)
    for idx in keep:
        out[lab == (idx + 1)] = 255

    # Dilatation légère pour compenser sous-segmentation
    out = cv2.dilate(out, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=1)
    return out


# Génération si besoin
img_files = sorted([p for p in IMAGES_DIR.glob('*') if p.is_file()])
msk_files = sorted([p for p in MASKS_DIR.glob('*') if p.is_file()])

if len(msk_files) == len(img_files) and len(img_files) > 0:
    print('Masks already present → supervised segmentation.')
else:
    print('No usable masks found → generating pseudo-masks...')
    # Clear masks dir
    for f in MASKS_DIR.glob('*'):
        if f.is_file():
            f.unlink()

    for i, p in enumerate(tqdm(img_files)):
        img = cv2.imread(p.as_posix(), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise RuntimeError(f'Failed to read image: {p}')
        m = generate_pseudo_mask(img)
        out = MASKS_DIR / f"mask_{i:06d}.png"
        cv2.imwrite(out.as_posix(), m)

    print('Generated masks:', len(list(MASKS_DIR.glob('*.png'))))


In [ ]:
import matplotlib.pyplot as plt

# Visualiser quelques exemples
n_show = 4
idxs = np.linspace(0, max(0, len(img_files) - 1), n_show, dtype=int)

plt.figure(figsize=(12, 3 * n_show))
for r, i in enumerate(idxs):
    img = cv2.imread(img_files[i].as_posix(), cv2.IMREAD_GRAYSCALE)
    msk = cv2.imread((MASKS_DIR / f"mask_{i:06d}.png").as_posix(), cv2.IMREAD_GRAYSCALE)
    if msk is None:
        # fallback if masks have different extension
        msk = cv2.imread(sorted(MASKS_DIR.glob('*'))[i].as_posix(), cv2.IMREAD_GRAYSCALE)

    plt.subplot(n_show, 2, 2*r + 1)
    plt.title(f'Image {i}')
    plt.imshow(img, cmap='gray')
    plt.axis('off')

    plt.subplot(n_show, 2, 2*r + 2)
    plt.title('Masque')
    plt.imshow(msk, cmap='gray')
    plt.axis('off')

plt.tight_layout()


## 3) Entraîner U‑Net (segmentation) avec augmentation

Cette étape entraîne un modèle de segmentation sur **tout le dataset** après téléchargement.
- Augmentations (Albumentations): variations luminosité/contraste, bruit, blur, petites rotations/shift/scale, etc.
- Sauvegarde du meilleur modèle sur validation.


In [ ]:
import random
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Hyperparamètres (ajustables)
IMG_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3
VAL_RATIO = 0.15
NUM_WORKERS = 2

MODELS_DIR = DATA_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
UNET_CKPT = MODELS_DIR / 'unet_best.pth'


def build_transforms(train: bool):
    if train:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.10, rotate_limit=8, border_mode=cv2.BORDER_REFLECT_101, p=0.7),
            A.RandomBrightnessContrast(p=0.7),
            A.RandomGamma(p=0.3),
            A.GaussNoise(var_limit=(5.0, 30.0), p=0.4),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.Normalize(mean=(0.5,), std=(0.25,)),
            ToTensorV2(transpose_mask=True),
        ])
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.5,), std=(0.25,)),
        ToTensorV2(transpose_mask=True),
    ])


class WeldSegDataset(Dataset):
    def __init__(self, img_paths, mask_paths, train: bool):
        self.img_paths = list(img_paths)
        self.mask_paths = list(mask_paths)
        self.tf = build_transforms(train=train)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        ip = self.img_paths[idx]
        mp = self.mask_paths[idx]

        img = cv2.imread(ip.as_posix(), cv2.IMREAD_GRAYSCALE)
        msk = cv2.imread(mp.as_posix(), cv2.IMREAD_GRAYSCALE)
        if img is None or msk is None:
            raise RuntimeError(f'Failed to read: {ip} / {mp}')

        # (H,W,1)
        img = img[..., None]
        msk = (msk > 127).astype(np.uint8)[..., None]

        out = self.tf(image=img, mask=msk)
        x = out['image'].float()      # (1,H,W)
        y = out['mask'].float()       # (1,H,W)
        return x, y


# Align images/masks by index naming mask_XXXXXX
img_paths = sorted([p for p in IMAGES_DIR.glob('*') if p.is_file()])
mask_paths = [MASKS_DIR / f"mask_{i:06d}.png" for i in range(len(img_paths))]

missing = [p for p in mask_paths if not p.exists()]
if missing:
    raise RuntimeError(f'Missing masks (expected mask_XXXXXX.png). Example missing: {missing[0]}')

n = len(img_paths)
idx = np.arange(n)
np.random.shuffle(idx)
val_n = max(1, int(n * VAL_RATIO))
val_idx = idx[:val_n]
tr_idx = idx[val_n:]

train_ds = WeldSegDataset([img_paths[i] for i in tr_idx], [mask_paths[i] for i in tr_idx], train=True)
val_ds   = WeldSegDataset([img_paths[i] for i in val_idx], [mask_paths[i] for i in val_idx], train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print('Train:', len(train_ds), 'Val:', len(val_ds))


In [ ]:
# --- U-Net training ---

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=1,
    classes=1,
    activation=None,
).to(DEVICE)

bce = nn.BCEWithLogitsLoss()

def dice_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2,3))
    den = (probs + targets).sum(dim=(2,3)) + eps
    dice = (num + eps) / den
    return 1 - dice.mean()


def iou_score(logits, targets, thr=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > thr).float()
    inter = (preds * targets).sum(dim=(2,3))
    union = (preds + targets - preds * targets).sum(dim=(2,3))
    iou = (inter + eps) / (union + eps)
    return iou.mean().item()


opt = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)

best_iou = -1.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0

    for x, y in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [train]'):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss = bce(logits, y) + dice_loss(logits, y)
        loss.backward()
        opt.step()
        tr_loss += loss.item() * x.size(0)

    tr_loss /= max(1, len(train_ds))

    model.eval()
    va_loss = 0.0
    va_iou = 0.0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} [val]'):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            logits = model(x)
            loss = bce(logits, y) + dice_loss(logits, y)
            va_loss += loss.item() * x.size(0)
            va_iou += iou_score(logits, y) * x.size(0)

    va_loss /= max(1, len(val_ds))
    va_iou /= max(1, len(val_ds))

    scheduler.step(va_iou)

    print(f"Epoch {epoch}: train_loss={tr_loss:.4f}  val_loss={va_loss:.4f}  val_iou={va_iou:.4f}")

    if va_iou > best_iou:
        best_iou = va_iou
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'val_iou': best_iou}, UNET_CKPT)
        print('  ✓ saved:', UNET_CKPT)

print('Best val IoU:', best_iou)


In [ ]:
# --- Visualisation rapide des prédictions ---
ckpt = torch.load(UNET_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print('Loaded best model from epoch', ckpt.get('epoch'), 'val_iou', ckpt.get('val_iou'))

@torch.no_grad()
def predict_mask_gray(gray: np.ndarray):
    # gray: (H,W) uint8
    tf = build_transforms(train=False)
    x = tf(image=gray[..., None], mask=np.zeros_like(gray)[..., None])['image'].unsqueeze(0).to(DEVICE)
    logits = model(x)
    prob = torch.sigmoid(logits)[0,0].detach().cpu().numpy()
    return prob

n_show = 4
idxs = np.linspace(0, max(0, len(img_paths) - 1), n_show, dtype=int)

plt.figure(figsize=(14, 4 * n_show))
for r, i in enumerate(idxs):
    gray = cv2.imread(img_paths[i].as_posix(), cv2.IMREAD_GRAYSCALE)
    gt = cv2.imread((MASKS_DIR / f"mask_{i:06d}.png").as_posix(), cv2.IMREAD_GRAYSCALE)
    prob = predict_mask_gray(gray)
    pred = (prob > 0.5).astype(np.uint8) * 255

    plt.subplot(n_show, 3, 3*r + 1)
    plt.title('Image')
    plt.imshow(gray, cmap='gray'); plt.axis('off')

    plt.subplot(n_show, 3, 3*r + 2)
    plt.title('Masque (GT / pseudo)')
    plt.imshow(gt, cmap='gray'); plt.axis('off')

    plt.subplot(n_show, 3, 3*r + 3)
    plt.title('Prédiction U-Net')
    plt.imshow(pred, cmap='gray'); plt.axis('off')

plt.tight_layout()


## 4) Analyse: largeur, continuité, homogénéité

À partir du masque (prédit), on calcule:
- **Largeur**: via transformée de distance + squelette (médiane des rayons ×2)
- **Continuité**: proportion de la longueur du squelette appartenant à la plus grande composante
- **Homogénéité**: variance/contraste interne le long de la soudure → score + labels pour CNN

> Si vous avez des métadonnées caméra (ex: mm/pixel), renseignez `PIXEL_SIZE_MM` pour obtenir la largeur en mm.


In [ ]:
from skimage.morphology import skeletonize
from skimage.measure import label

PIXEL_SIZE_MM = None  # ex: 0.02 (mm/pixel) si connu


def weld_metrics(gray: np.ndarray, mask_bin: np.ndarray):
    """Calcule largeur, continuité, homogénéité (scores) sur une image.

    Args:
      gray: (H,W) uint8
      mask_bin: (H,W) bool/0-1/0-255

    Returns dict.
    """
    m = mask_bin
    if m.dtype != np.bool_:
        m = m > 0

    if m.sum() < 10:
        return {
            'width_px_median': 0.0,
            'width_mm_median': None,
            'continuity_score': 0.0,
            'homogeneity_score': 0.0,
            'intensity_mean': float(gray.mean()),
            'intensity_std_in_weld': None,
        }

    # Skeleton
    sk = skeletonize(m).astype(np.uint8)
    sk_len = int(sk.sum())

    # Continuity: largest connected component ratio
    lab = label(sk, connectivity=2)
    if lab.max() <= 0:
        continuity = 0.0
    else:
        sizes = [(lab == i).sum() for i in range(1, lab.max() + 1)]
        continuity = float(max(sizes) / max(1, sk_len))

    # Width: distance transform sampled on skeleton
    dist = cv2.distanceTransform((m.astype(np.uint8) * 255), distanceType=cv2.DIST_L2, maskSize=5)
    radii = dist[sk.astype(bool)]
    width_px = float(2.0 * np.median(radii)) if radii.size else 0.0
    width_mm = float(width_px * PIXEL_SIZE_MM) if PIXEL_SIZE_MM is not None else None

    # Homogeneity: intensity variation inside weld
    weld_vals = gray[m].astype(np.float32)
    mu = float(weld_vals.mean())
    sigma = float(weld_vals.std())
    cv = float(sigma / (mu + 1e-6))

    # Score in [0,1] (heuristique): plus CV est faible, plus homogène
    homogeneity = float(np.exp(-3.0 * cv))

    return {
        'width_px_median': width_px,
        'width_mm_median': width_mm,
        'continuity_score': continuity,
        'homogeneity_score': homogeneity,
        'intensity_mean': float(gray.mean()),
        'intensity_std_in_weld': sigma,
        'skeleton_length_px': sk_len,
    }


# Exemple sur une image du set de validation
example_i = int(val_idx[0]) if len(val_idx) else 0
gray = cv2.imread(img_paths[example_i].as_posix(), cv2.IMREAD_GRAYSCALE)
prob = predict_mask_gray(gray)
pred = (prob > 0.5).astype(np.uint8)

metrics = weld_metrics(gray, pred)
metrics


## 5) CNN simple: homogène vs non homogène

On entraîne un petit CNN sur des **patchs** extraits le long de la soudure.
- Labels **pseudo**: un patch est *homogène* si la variance locale (ou écart-type) dans la zone de soudure est faible.
- Sortie: probabilité `p(homogène)`.

> Si vous disposez de labels humains (défauts / non défauts), remplacez la génération de labels par vos annotations.


In [ ]:
# --- Build patch dataset for homogeneity classifier ---
import torch.nn.functional as F

PATCH_SIZE = 64
PATCHES_PER_IMAGE = 50   # ajuster selon taille dataset
MAX_PATCHES_TOTAL = 20000


def extract_patches(gray: np.ndarray, mask_bin: np.ndarray, n_patches: int):
    """Extrait des patchs centrés sur le squelette de la soudure.

    Returns:
      patches: (N,1,H,W) float32 in [0,1]
      local_std: (N,) float32 (proxy défaut/homogénéité)
    """
    m = (mask_bin > 0).astype(np.uint8)
    sk = skeletonize(m.astype(bool)).astype(np.uint8)
    ys, xs = np.where(sk > 0)
    if len(xs) == 0:
        return np.zeros((0, 1, PATCH_SIZE, PATCH_SIZE), np.float32), np.zeros((0,), np.float32)

    # sous-échantillonner des points
    sel = np.random.choice(len(xs), size=min(n_patches, len(xs)), replace=False)
    xs = xs[sel]
    ys = ys[sel]

    pad = PATCH_SIZE // 2
    gpad = np.pad(gray.astype(np.float32) / 255.0, ((pad, pad), (pad, pad)), mode='reflect')
    mpad = np.pad(m.astype(np.uint8), ((pad, pad), (pad, pad)), mode='reflect')

    patches = []
    stds = []
    for x, y in zip(xs, ys):
        x0, x1 = x, x + 2*pad
        y0, y1 = y, y + 2*pad
        patch = gpad[y0:y1, x0:x1]
        mpatch = mpad[y0:y1, x0:x1]

        if patch.shape != (PATCH_SIZE, PATCH_SIZE):
            continue

        # std sur pixels dans la zone soudure; fallback sur patch entier
        if mpatch.sum() > 20:
            vals = patch[mpatch > 0]
        else:
            vals = patch.reshape(-1)
        std = float(vals.std())

        patches.append(patch[None, ...])  # (1,H,W)
        stds.append(std)

    if not patches:
        return np.zeros((0, 1, PATCH_SIZE, PATCH_SIZE), np.float32), np.zeros((0,), np.float32)

    return np.stack(patches, axis=0).astype(np.float32), np.array(stds, np.float32)


# Construire patches + pseudo-labels
all_patches = []
all_stds = []

# Pour accélérer, on utilise les masques du dataset (pseudo ou GT) au lieu de prédire partout.
# Option: remplacez mask_bin par la prédiction U-Net si vous voulez des labels plus cohérents.
for i in tqdm(range(len(img_paths)), desc='Extracting patches'):
    gray = cv2.imread(img_paths[i].as_posix(), cv2.IMREAD_GRAYSCALE)
    msk = cv2.imread((MASKS_DIR / f"mask_{i:06d}.png").as_posix(), cv2.IMREAD_GRAYSCALE)
    if gray is None or msk is None:
        continue
    patches, stds = extract_patches(gray, msk, PATCHES_PER_IMAGE)
    if patches.shape[0] == 0:
        continue
    all_patches.append(patches)
    all_stds.append(stds)

    if sum(p.shape[0] for p in all_patches) >= MAX_PATCHES_TOTAL:
        break

X = np.concatenate(all_patches, axis=0)
stds = np.concatenate(all_stds, axis=0)

# Seuil: par défaut, les 70% plus faibles std → homogène
STD_QUANTILE = 0.70
thr = float(np.quantile(stds, STD_QUANTILE))
y = (stds <= thr).astype(np.int64)

print('Patches:', X.shape, 'Homogeneous ratio:', float(y.mean()), 'Std thr:', thr)


In [ ]:
# --- Train simple CNN classifier ---

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, 2)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class PatchDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]


# Split
n = X.shape[0]
idx = np.arange(n)
np.random.shuffle(idx)
val_n = max(1, int(0.15 * n))
val_idx2 = idx[:val_n]
tr_idx2 = idx[val_n:]

tr_ds2 = PatchDataset(X[tr_idx2], y[tr_idx2])
va_ds2 = PatchDataset(X[val_idx2], y[val_idx2])

tr_dl2 = DataLoader(tr_ds2, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
va_dl2 = DataLoader(va_ds2, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

clf = SimpleCNN().to(DEVICE)
opt2 = torch.optim.AdamW(clf.parameters(), lr=1e-3)

best_acc = -1.0
CLF_CKPT = MODELS_DIR / 'cnn_homogeneity_best.pth'

for epoch in range(1, 6):
    clf.train()
    tr_loss = 0.0
    for xb, yb in tqdm(tr_dl2, desc=f'CNN epoch {epoch}/5 [train]'):
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        opt2.zero_grad(set_to_none=True)
        logits = clf(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        opt2.step()
        tr_loss += loss.item() * xb.size(0)

    tr_loss /= max(1, len(tr_ds2))

    clf.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in tqdm(va_dl2, desc=f'CNN epoch {epoch}/5 [val]'):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            logits = clf(xb)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.numel()

    acc = correct / max(1, total)
    print(f'CNN epoch {epoch}: train_loss={tr_loss:.4f} val_acc={acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        torch.save({'model': clf.state_dict(), 'val_acc': best_acc}, CLF_CKPT)
        print('  ✓ saved:', CLF_CKPT)

print('Best val acc:', best_acc)


## 6) Test interactif (image / vidéo)

- Upload une **image** ou une **vidéo**.
- Le notebook produit:
  - masque segmenté + overlay
  - **largeur**, **continuité**, **homogénéité**
  - score qualité (heuristique)


In [ ]:
import io
from IPython.display import display, clear_output
import ipywidgets as widgets
import torch.nn as nn

# Load CNN checkpoint (if trained)
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, 2)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


clf = SimpleCNN().to(DEVICE)
clf_loaded = False
ckpt_path = MODELS_DIR / 'cnn_homogeneity_best.pth'
if ckpt_path.exists():
    ck = torch.load(ckpt_path, map_location=DEVICE)
    clf.load_state_dict(ck['model'])
    clf.eval()
    clf_loaded = True
    print('Loaded homogeneity CNN. val_acc:', ck.get('val_acc'))
else:
    print('Homogeneity CNN not found (run section 5 to train it).')


def overlay_mask(gray: np.ndarray, mask01: np.ndarray, alpha=0.45):
    """Overlay rouge sur image grayscale."""
    g = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    red = np.zeros_like(g)
    red[..., 2] = 255
    m = (mask01 > 0).astype(np.float32)[..., None]
    out = (g.astype(np.float32) * (1 - alpha*m) + red.astype(np.float32) * (alpha*m)).astype(np.uint8)
    return out


@torch.no_grad()
def homogeneity_prob_from_mask(gray: np.ndarray, mask01: np.ndarray):
    """p(homogène) moyen sur patchs le long de la soudure."""
    if not clf_loaded:
        return None

    patches, _ = extract_patches(gray, (mask01 > 0).astype(np.uint8) * 255, n_patches=80)
    if patches.shape[0] == 0:
        return None

    xb = torch.from_numpy(patches).to(DEVICE)
    logits = clf(xb)
    probs = torch.softmax(logits, dim=1)[:, 1]  # class 1 = homogène
    return float(probs.mean().detach().cpu().numpy())


from typing import Optional

def quality_score(metrics: dict, p_hom: Optional[float]):
    # Heuristique simple: continuité × homogénéité × pénalité si largeur nulle
    cont = float(metrics.get('continuity_score', 0.0) or 0.0)
    homog = float(metrics.get('homogeneity_score', 0.0) or 0.0)
    if p_hom is not None:
        homog = 0.5 * homog + 0.5 * float(p_hom)
    width = float(metrics.get('width_px_median', 0.0) or 0.0)
    width_ok = 1.0 if width > 1.0 else 0.5
    return float(np.clip(cont * homog * width_ok, 0.0, 1.0))


uploader = widgets.FileUpload(accept='.png,.jpg,.jpeg,.bmp,.tif,.tiff,.mp4,.avi,.mov,.mkv', multiple=False)
thr = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description='Seuil')
max_frames = widgets.IntSlider(value=200, min=20, max=2000, step=20, description='Frames')
stride = widgets.IntSlider(value=3, min=1, max=20, step=1, description='Stride')
run_btn = widgets.Button(description='Run', button_style='primary')
out = widgets.Output()


def _decode_uploaded(data: bytes):
    arr = np.frombuffer(data, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
    return img


def run_inference_on_image(gray: np.ndarray, threshold: float):
    prob = predict_mask_gray(gray)
    mask01 = (prob > threshold).astype(np.uint8)
    met = weld_metrics(gray, mask01)
    p_hom = homogeneity_prob_from_mask(gray, mask01)
    q = quality_score(met, p_hom)

    vis = overlay_mask(gray, mask01)
    return vis, mask01, met, p_hom, q


def _handle_click(_):
    with out:
        clear_output()
        if len(uploader.value) == 0:
            print('Upload a file first.')
            return

        item = next(iter(uploader.value.values()))
        name = item['metadata']['name']
        data = item['content']
        ext = name.lower().split('.')[-1]

        if ext in {'mp4','avi','mov','mkv'}:
            in_path = '/content/input_video.' + ext
            with open(in_path, 'wb') as f:
                f.write(data)

            cap = cv2.VideoCapture(in_path)
            if not cap.isOpened():
                raise RuntimeError('Cannot open video')

            fps = cap.get(cv2.CAP_PROP_FPS) or 25
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

            out_path = '/content/output_annotated.mp4'
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            vw = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

            frame_id = 0
            kept = 0
            last_metrics = None

            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                if frame_id % int(stride.value) != 0:
                    frame_id += 1
                    continue

                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                vis, _, met, p_hom, q = run_inference_on_image(gray, float(thr.value))

                # Annotate
                txt = f"width_px={met['width_px_median']:.1f} cont={met['continuity_score']:.2f} homog={met['homogeneity_score']:.2f}"
                if p_hom is not None:
                    txt += f" cnn_p={p_hom:.2f}"
                txt += f" Q={q:.2f}"

                vis_bgr = vis
                cv2.putText(vis_bgr, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2, cv2.LINE_AA)
                vw.write(vis_bgr)

                last_metrics = (met, p_hom, q)
                kept += 1
                frame_id += 1
                if kept >= int(max_frames.value):
                    break

            cap.release()
            vw.release()

            print('Saved:', out_path)
            if last_metrics is not None:
                met, p_hom, q = last_metrics
                print('Last-frame metrics:', met)
                print('CNN p(homog):', p_hom)
                print('Quality score:', q)

            from google.colab import files
            files.download(out_path)
            return

        # Image
        gray = _decode_uploaded(data)
        if gray is None:
            raise RuntimeError('Cannot decode image')

        vis, mask01, met, p_hom, q = run_inference_on_image(gray, float(thr.value))

        print('Metrics:', met)
        print('CNN p(homog):', p_hom)
        print('Quality score:', q)

        plt.figure(figsize=(16, 5))
        plt.subplot(1,3,1); plt.title('Image'); plt.imshow(gray, cmap='gray'); plt.axis('off')
        plt.subplot(1,3,2); plt.title('Masque'); plt.imshow(mask01, cmap='gray'); plt.axis('off')
        plt.subplot(1,3,3); plt.title('Overlay'); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off')
        plt.tight_layout()


run_btn.on_click(_handle_click)

display(widgets.VBox([
    widgets.HTML('<b>Upload puis Run</b>'),
    uploader,
    widgets.HBox([thr, stride, max_frames]),
    run_btn,
    out,
]))


## 7) (Optionnel) Rapport rapide sur le dataset

Calcule des statistiques (largeur/continuité/homogénéité) sur un sous-ensemble pour aider le contrôle qualité global.


In [ ]:
# Quick report (sampled for speed)
N_REPORT = min(200, len(img_paths))
step = max(1, len(img_paths) // N_REPORT)
sel = list(range(0, len(img_paths), step))[:N_REPORT]

rows = []
for i in tqdm(sel, desc='Reporting'):
    gray = cv2.imread(img_paths[i].as_posix(), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        continue
    prob = predict_mask_gray(gray)
    mask01 = (prob > 0.5).astype(np.uint8)
    met = weld_metrics(gray, mask01)
    rows.append(met)

# Simple plots
widths = [r['width_px_median'] for r in rows]
conts = [r['continuity_score'] for r in rows]
homos = [r['homogeneity_score'] for r in rows]

plt.figure(figsize=(14,4))
plt.subplot(1,3,1); plt.title('Largeur (px)'); plt.hist(widths, bins=30)
plt.subplot(1,3,2); plt.title('Continuité'); plt.hist(conts, bins=30, range=(0,1))
plt.subplot(1,3,3); plt.title('Homogénéité'); plt.hist(homos, bins=30, range=(0,1))
plt.tight_layout()

print('Width median (px):', float(np.median(widths)) if widths else None)
print('Continuity mean:', float(np.mean(conts)) if conts else None)
print('Homogeneity mean:', float(np.mean(homos)) if homos else None)
